In [2]:
import sys
import subprocess
import importlib.util

packages = {
    "unsloth": "unsloth",
    "gradio": "gradio",
    "huggingface_hub": "huggingface_hub",
}
missing = [
    package for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", *missing
    ])
    print("Installation finished. Restart the Python kernel, then run Cell 2 onward.")
else:
    print("Packages are present. Continue to Cell 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uni

In [2]:
# Import Unsloth before Transformers, PEFT, or TRL.
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

import torch
import gradio as gr
from importlib.metadata import version

for package in ["unsloth", "torch", "transformers", "gradio"]:
    print(package, version(package))

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle settings, then restart the session.")

print("GPU:", torch.cuda.get_device_name(0))


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.10.7
torch 2.11.0+cu128
transformers 5.16.1
gradio 6.26.0
GPU: Tesla T4


In [2]:
from huggingface_hub import HfApi, login

MODEL_ID = "GAnirudh711/mistral-7b-dsa-expert-v2"

# Optional for a public repository; needed if private/gated access is required.
# In Kaggle, add a secret named HF_TOKEN and enable notebook access to it.
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No Kaggle HF_TOKEN available; trying public/cached access.")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face login complete.")

api = HfApi()
files = api.list_repo_files(MODEL_ID, repo_type="model")

if "adapter_config.json" not in files:
    raise RuntimeError("This repository does not contain adapter_config.json. Check MODEL_ID.")

if not any(
    name.rsplit("/", 1)[-1] in {"adapter_model.safetensors", "adapter_model.bin"}
    for name in files
):
    raise RuntimeError("Adapter weights were not found. Check that the checkpoint upload completed.")

print("Found the saved adapter configuration and weights:", MODEL_ID)


No Kaggle HF_TOKEN available; trying public/cached access.
Found the saved adapter configuration and weights: GAnirudh711/mistral-7b-dsa-expert-v2


In [4]:
# Run this once in a fresh kernel to avoid loading duplicate model copies.
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MODEL_ID = "GAnirudh711/mistral-7b-dsa-expert-v2"
CONTEXT_WINDOW = 8192
MAX_NEW_TOKENS = 1500
PROMPT_BUDGET = CONTEXT_WINDOW - MAX_NEW_TOKENS

assert 0 < MAX_NEW_TOKENS < CONTEXT_WINDOW

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=CONTEXT_WINDOW,
    dtype=None,
    load_in_4bit=True,
)

# This adapter was trained using ChatML: keep that format.
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = "You are an expert DSA (Data Structures & Algorithms) tutor and competitive programmer. Explain your approach and provide clean code."

STOP_IDS = [tokenizer.eos_token_id]
chatml_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if chatml_end is not None and chatml_end != tokenizer.unk_token_id:
    STOP_IDS.append(chatml_end)
STOP_IDS = sorted({value for value in STOP_IDS if value is not None})
if not STOP_IDS:
    raise RuntimeError("No valid end-of-answer token was found.")

print("Model ready. Total context:", CONTEXT_WINDOW)
print("Maximum formatted input:", PROMPT_BUDGET)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.7: Fast Mistral patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Unsloth 2026.10.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = </s>.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_yg4bidwf/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/_unsloth_sentencepiece_temp/tokenizer_yg4bidwf.


Model ready. Total context: 8192
Maximum formatted input: 6692


In [7]:
def extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "") if content.get("type") == "text" else ""
    if isinstance(content, list):
        return "\n".join(filter(None, (extract_text(item) for item in content)))
    return ""


def normalize_history(history):
    messages = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            text = extract_text(item.get("content", ""))
            if role in {"user", "assistant"} and text.strip():
                messages.append({"role": role, "content": text})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            for role, content in zip(("user", "assistant"), item):
                text = extract_text(content)
                if text.strip():
                    messages.append({"role": role, "content": text})
    return messages


def complete_exchanges(messages):
    exchanges = []
    pending = None
    for message in messages:
        if message["role"] == "user":
            pending = [message.copy()]
        elif pending is not None:
            if len(pending) == 1:
                pending.append(message.copy())
                exchanges.append(pending)
            else:
                pending[-1]["content"] += "\n" + message["content"]
    return exchanges


def encode_prompt(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )


def build_prompt(user_text, history):
    exchanges = complete_exchanges(normalize_history(history))
    removed = 0
    while True:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for exchange in exchanges:
            messages.extend(exchange)
        messages.append({"role": "user", "content": user_text})

        input_ids = encode_prompt(messages)
        if input_ids.shape[-1] <= PROMPT_BUDGET:
            return input_ids, removed

        if not exchanges:
            raise gr.Error("This question exceeds the input token budget. Shorten it.")
        exchanges.pop(0)
        removed += 1


def chat_with_tutor(user_message, history):
    user_text = extract_text(user_message).strip()
    if not user_text:
        raise gr.Error("Please enter a text question.")

    input_ids, removed = build_prompt(user_text, history)
    if removed:
        gr.Warning(f"Context full: omitted {removed} oldest exchange(s) for this answer.")

    input_ids = input_ids.to(model.get_input_embeddings().weight.device)
    attention_mask = torch.ones_like(input_ids)
    print(f"Input tokens: {input_ids.shape[-1]} / {PROMPT_BUDGET}")

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
           do_sample=True,
        temperature=0.2,
        repetition_penalty=1.1,
            use_cache=True,
            eos_token_id=STOP_IDS,
            pad_token_id=(
                tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else STOP_IDS[0]
            ),
        )

    answer_ids = output_ids[0, input_ids.shape[-1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()


In [9]:
# Compare the same question with and without your trained adapter.
# No training, new model download, or second model copy is required.
import torch
prompt = """
Explain Kadane's algorithm for the maximum sum of a
NON-EMPTY contiguous subarray of a non-empty integer array.

Explain how it handles arrays where every element is negative.
Provide C++17 code and time and space complexity.

Trace your algorithm on:
1. [-5, -2, -6]
2. [-2, 1, -3, 4, -1, 2, 1, -5, 4]
""".strip()

if not callable(getattr(model, "disable_adapter", None)):
    raise RuntimeError(
        "This loaded model does not expose disable_adapter(). "
        "Please share this error before changing other cells."
    )

print("=" * 60)
print("A: YOUR FINE-TUNED MODEL — ADAPTER ENABLED")
print("=" * 60)

fine_tuned_answer = chat_with_tutor(
    prompt,
    history=[],
)
print(fine_tuned_answer)

print("\n" + "=" * 60)
print("B: SAME MODEL — ADAPTER TEMPORARILY DISABLED")
print("=" * 60)

with model.disable_adapter():
    base_answer = chat_with_tutor(
        prompt,
        history=[],
    )

print(base_answer)

print("\nComparison finished. Your adapter is enabled again.")

A: YOUR FINE-TUNED MODEL — ADAPTER ENABLED
Input tokens: 170 / 6692


Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```cpp
int maxSubArray(vector<int>& nums) {
    int n = nums.size();
    if (n == 0) return 0;
    
    int currMax = nums[0];
    int globalMax = nums[0];
    
    for (int i = 1; i < n; ++i) {
        currMax = max(nums[i], currMax + nums[i]);
        globalMax = max(globalMax, currMax);
    }
    
    return globalMax;
}
```

B: SAME MODEL — ADAPTER TEMPORARILY DISABLED
Input tokens: 170 / 6692
Kadane's algorithm, also known as Maximum Subarray Problem, finds the maximum sum of a contiguous subarray in a given one-dimensional integer array. It works by maintaining two variables: `max_so_far` and `current`. The former keeps track of the maximum sum found so far, while the latter stores the current sum of the elements in the subarray being considered.

The algorithm iterates through the input array from left to right, updating `current` with the sum of the current and next element if the sum is positive; otherwise, it sets `current` to zero. At each step, it updates `max_so_far` with 

In [10]:
from transformers import AutoTokenizer

# Separate tokenizer: leave your fine-tuned chat tokenizer unchanged.
native_tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3"
)

test_prompt = """
Explain Kadane's algorithm for the maximum sum of a
NON-EMPTY contiguous subarray of a non-empty integer array.

Provide C++17 code, explain the initialization, and handle
all-negative arrays.

Calculate the result for:
1. [-5, -2, -6]
2. [-2, 1, -3, 4, -1, 2, 1, -5, 4]
3. [1, 2]

Make sure the explanation and example results agree with the code.
""".strip()

# Use the base model's own template instead of ChatML.
native_prompt = native_tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    tokenize=False,
    add_generation_prompt=True,
)

native_inputs = native_tokenizer(
    native_prompt,
    add_special_tokens=False,
    return_tensors="pt",
)

device = model.get_input_embeddings().weight.device
native_inputs = {
    key: value.to(device)
    for key, value in native_inputs.items()
}

print("BASE MODEL: NATIVE MISTRAL FORMAT, ADAPTER DISABLED")

with model.disable_adapter():
    with torch.inference_mode():
        native_outputs = model.generate(
            **native_inputs,
            max_new_tokens=1500,
            do_sample=True,
            temperature=0.2,
            repetition_penalty=1.1,
            use_cache=True,
            eos_token_id=native_tokenizer.eos_token_id,
            pad_token_id=native_tokenizer.eos_token_id,
        )

new_tokens = native_outputs[
    0,
    native_inputs["input_ids"].shape[-1]:,
]

print(
    native_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )
)

print("\nFinished. Your trained adapter is enabled again.")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL: NATIVE MISTRAL FORMAT, ADAPTER DISABLED
Kadane's algorithm, also known as the Maximum Subarray Problem, is used to find the maximum sum of a contiguous subarray within a one-dimensional array of integers. The algorithm works by maintaining two variables: `current` (to store the current sum) and `maximum` (to store the maximum sum found so far).

The algorithm iterates through the input array, updating the `current` variable based on the sum of the current element and the previous `current` value or zero if starting a new subarray. At each step, it updates the `maximum` variable with the greater value between the current `maximum` and the `current` sum.

Here's the C++17 implementation of Kadane's algorithm:

```cpp
#include <iostream>
#include <numeric>
#include <vector>

int kadanesAlgorithm(const std::vector<int>& arr) {
    int n = static_cast<int>(arr.size());
    int maxCurrent = 0;
    int maxGlobal = std::accumulate(arr.begin(), arr.end(), 0); // Initialize maxGlobal

In [13]:
import gradio as gr
gr.close_all()

demo = gr.ChatInterface(
    fn=chat_with_tutor,
    title="DSA Expert AI Tutor",
    description="Your existing fine-tuned model with an 8K token context setting.",
    examples=[
        "Explain Kadane's algorithm for maximum subarray sum.",
        "Explain trapping rain water using two pointers.",
        "What is the difference between substring and subsequence?",
    ],
    concurrency_limit=1,
)

demo.queue().launch(share=True, show_error=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4aa7459a445441cc7f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: 68 / 6692


Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: 333 / 6692


...

Input tokens: 1903 / 6692


Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: 2538 / 6692


Both `max_new_tokens` (=1500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: 2968 / 6692


In [5]:
import torch, re, subprocess, sys, tempfile, os, json

SYS = "You are an expert DSA (Data Structures & Algorithms) tutor and competitive programmer. Explain your approach and provide clean code."
T = []
def add(name, desc, *tests):
    T.append((name, f"Write a Python function `{name}` that {desc} Put the code in a ```python block.", "\n".join(tests)))

add("two_sum(nums, target)", "returns the indices [i, j] with i < j of the two numbers adding up to target. Exactly one solution exists.",
    "assert two_sum([2,7,11,15],9)==[0,1]", "assert two_sum([3,2,4],6)==[1,2]", "assert two_sum([3,3],6)==[0,1]")
add("max_subarray(nums)", "returns the largest sum of a non-empty contiguous subarray.",
    "assert max_subarray([-2,1,-3,4,-1,2,1,-5,4])==6", "assert max_subarray([-5,-2,-6])==-2", "assert max_subarray([1])==1", "assert max_subarray([5,4,-1,7,8])==23")
add("binary_search(arr, target)", "returns the index of target in the sorted list arr (all elements unique), or -1 if absent.",
    "assert binary_search([1,3,5,7,9],5)==2", "assert binary_search([1,3,5,7,9],4)==-1", "assert binary_search([],1)==-1", "assert binary_search([2],2)==0")
add("is_palindrome(s)", "returns True if s is a palindrome considering only alphanumeric characters and ignoring case.",
    "assert is_palindrome('A man, a plan, a canal: Panama')==True", "assert is_palindrome('race a car')==False", "assert is_palindrome('')==True", "assert is_palindrome('0P')==False")
add("valid_parentheses(s)", "returns True if the string of brackets ()[]{} is valid.",
    "assert valid_parentheses('()[]{}')==True", "assert valid_parentheses('(]')==False", "assert valid_parentheses('([)]')==False", "assert valid_parentheses('{[]}')==True", "assert valid_parentheses('')==True")
add("fibonacci(n)", "returns the n-th Fibonacci number with fibonacci(0)=0 and fibonacci(1)=1. It must run fast for n=50.",
    "assert fibonacci(0)==0", "assert fibonacci(1)==1", "assert fibonacci(10)==55", "assert fibonacci(50)==12586269025")
add("merge_intervals(intervals)", "merges overlapping intervals (touching ones like [1,4] and [4,5] merge) and returns them sorted by start as a list of [start, end] lists.",
    "assert merge_intervals([[1,3],[2,6],[8,10],[15,18]])==[[1,6],[8,10],[15,18]]", "assert merge_intervals([[1,4],[4,5]])==[[1,5]]", "assert merge_intervals([])==[]", "assert merge_intervals([[5,6],[1,2]])==[[1,2],[5,6]]")
add("climb_stairs(n)", "returns the number of distinct ways to climb n stairs taking 1 or 2 steps at a time (n >= 1).",
    "assert climb_stairs(1)==1", "assert climb_stairs(2)==2", "assert climb_stairs(5)==8", "assert climb_stairs(10)==89")
add("longest_common_prefix(strs)", "returns the longest common prefix of a list of strings, or '' if there is none or the list is empty.",
    "assert longest_common_prefix(['flower','flow','flight'])=='fl'", "assert longest_common_prefix(['dog','racecar','car'])==''", "assert longest_common_prefix(['a'])=='a'", "assert longest_common_prefix([])==''")
add("rotate_array(nums, k)", "returns a NEW list with nums rotated to the right by k steps (k may be larger than len(nums)).",
    "assert rotate_array([1,2,3,4,5,6,7],3)==[5,6,7,1,2,3,4]", "assert rotate_array([1,2],3)==[2,1]", "assert rotate_array([1],0)==[1]")
add("product_except_self(nums)", "returns a list where element i is the product of all elements except nums[i], without using division.",
    "assert product_except_self([1,2,3,4])==[24,12,8,6]", "assert product_except_self([-1,1,0,-3,3])==[0,0,9,0,0]")
add("top_k_frequent(nums, k)", "returns the k most frequent elements in ascending order. The answer is unique.",
    "assert top_k_frequent([1,1,1,2,2,3],2)==[1,2]", "assert top_k_frequent([1],1)==[1]", "assert top_k_frequent([4,4,4,5,5,6,6,6,6],2)==[4,6]")
add("num_islands(grid)", "returns the number of islands in a grid (list of lists of the strings '1' and '0'); islands connect horizontally or vertically.",
    "assert num_islands([['1','1','0'],['0','1','0'],['0','0','1']])==2", "assert num_islands([['1']])==1", "assert num_islands([['0']])==0",
    "assert num_islands([['1','1','1','1','0'],['1','1','0','1','0'],['1','1','0','0','0'],['0','0','0','0','0']])==1")
add("coin_change(coins, amount)", "returns the fewest coins needed to make amount, or -1 if impossible. Coins can be reused.",
    "assert coin_change([1,2,5],11)==3", "assert coin_change([2],3)==-1", "assert coin_change([1],0)==0", "assert coin_change([186,419,83,408],6249)==20")
add("lis_length(nums)", "returns the length of the longest strictly increasing subsequence (0 for an empty list).",
    "assert lis_length([10,9,2,5,3,7,101,18])==4", "assert lis_length([0,1,0,3,2,3])==4", "assert lis_length([7,7,7])==1", "assert lis_length([])==0")
add("dijkstra(n, edges, src)", "takes n nodes labeled 0..n-1 and a list of directed edges [u, v, w] with non-negative weights, and returns a list of shortest distances from src, using -1 for unreachable nodes.",
    "assert dijkstra(5,[[0,1,4],[0,2,1],[2,1,2],[1,3,1],[2,3,5]],0)==[0,3,1,4,-1]", "assert dijkstra(1,[],0)==[0]", "assert dijkstra(3,[[0,1,5]],0)==[0,5,-1]")
add("has_cycle(n, edges)", "returns True if the directed graph with nodes 0..n-1 and edges [u, v] contains a cycle (a self-loop counts as a cycle), else False.",
    "assert has_cycle(2,[[0,1],[1,0]])==True", "assert has_cycle(3,[[0,1],[1,2]])==False", "assert has_cycle(1,[[0,0]])==True", "assert has_cycle(4,[[0,1],[1,2],[2,3],[3,1]])==True", "assert has_cycle(3,[])==False")
add("kth_largest(nums, k)", "returns the k-th largest element, counting duplicates.",
    "assert kth_largest([3,2,1,5,6,4],2)==5", "assert kth_largest([3,2,3,1,2,4,5,5,6],4)==4", "assert kth_largest([1],1)==1")
add("length_of_longest_substring(s)", "returns the length of the longest substring without repeating characters.",
    "assert length_of_longest_substring('abcabcbb')==3", "assert length_of_longest_substring('bbbbb')==1", "assert length_of_longest_substring('pwwkew')==3", "assert length_of_longest_substring('')==0", "assert length_of_longest_substring('dvdf')==3")
add("trap_rain_water(height)", "returns the total units of rain water trapped between bars of the given heights.",
    "assert trap_rain_water([0,1,0,2,1,0,1,3,2,1,2,1])==6", "assert trap_rain_water([4,2,0,3,2,5])==9", "assert trap_rain_water([])==0", "assert trap_rain_water([3])==0")

def generate(user_msg, adapter, max_new_tokens=900):
    msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": user_msg}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=True).to(model.get_input_embeddings().weight.device)
    kw = dict(max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=STOP_IDS, pad_token_id=tokenizer.eos_token_id)
    with torch.no_grad():
        if adapter:
            out = model.generate(**enc, **kw)
        else:
            with model.disable_adapter():
                out = model.generate(**enc, **kw)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

def extract_code(ans, fname):
    blocks = re.findall(r"```(?:python|py)?\n(.*?)```", ans, flags=re.S)
    tail = re.search(r"```(?:python|py)?\n(.*)$", ans, flags=re.S)
    if tail: blocks.append(tail.group(1))
    for b in blocks:
        if f"def {fname}" in b:
            return b
    return None

def run_tests(code, tests, timeout=10):
    src = code + "\n\n" + tests + "\nprint('PASS')\n"
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(src); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=timeout, stdin=subprocess.DEVNULL)
        ok = r.returncode == 0 and "PASS" in r.stdout
        err = (r.stderr.strip().splitlines() or [""])[-1][:80]
    except subprocess.TimeoutExpired:
        ok, err = False, "timeout"
    finally:
        os.unlink(path)
    return ok, err

results = []
for sig, prompt, tests in T:
    fname = sig.split("(")[0]
    row = {"task": fname}
    for label, use_adapter in [("base", False), ("v2", True)]:
        ans = generate(prompt, use_adapter)
        code = extract_code(ans, fname)
        ok, err = run_tests(code, tests) if code else (False, "no code block")
        row[label], row[label + "_err"], row[label + "_answer"] = ok, err, ans
    results.append(row)
    print(f"{fname:30s} base={'PASS' if row['base'] else 'FAIL':4s}  v2={'PASS' if row['v2'] else 'FAIL':4s}  {row['v2_err'] if not row['v2'] else ''}")

b = sum(r["base"] for r in results); v = sum(r["v2"] for r in results)
print(f"\nBase model: {b}/{len(T)}   |   v2 fine-tuned: {v}/{len(T)}")
json.dump(results, open("eval_results.json", "w"), indent=1)

Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


two_sum                        base=PASS  v2=PASS  


Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


max_subarray                   base=PASS  v2=PASS  


Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

kth_largest                    base=PASS  v2=PASS  


Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


length_of_longest_substring    base=PASS  v2=FAIL  AssertionError


Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


trap_rain_water                base=PASS  v2=PASS  

Base model: 12/20   |   v2 fine-tuned: 15/20
